# BEV Lab 2 — BEVFusion: Camera + LiDAR Fusion

In Lab 1 we ran **LSS** — a camera-only model that lifts 2D image features into a 3D frustum and splats them onto a BEV grid.

In this lab we add **LiDAR**. BEVFusion (Liu et al., 2022) builds a BEV representation from each modality independently, then fuses them by concatenation:

```
Camera images  →  LSS-style lift+splat  →  Camera BEV  ─┐
                                                         ├─ concat → conv head → 3D boxes / BEV seg
LiDAR points   →  VoxelNet / PointPillars →  LiDAR BEV  ─┘
```

We'll load three pretrained checkpoints on the same nuScenes keyframe:
1. **Camera-only** — what does the camera see alone?
2. **LiDAR-only** — what does the LiDAR see alone?
3. **Fused (camera + LiDAR)** — what does combining both give you?

> Reference: *BEVFusion: Multi-Task Multi-Sensor Fusion with Unified Bird's-Eye View Representation* — Liu et al., ICRA 2023.  
> Code: [mit-han-lab/bevfusion](https://github.com/mit-han-lab/bevfusion)

## Part 1 — Setup

In [ ]:
# Clone BEVFusion (we only use its configs + model code, NOT setup.py)
!git clone -q https://github.com/mit-han-lab/bevfusion.git

# Install mmcv-full with the right wheel for YOUR torch + CUDA version
import torch
cuda_ver = torch.version.cuda.replace(".", "")[:3]
torch_ver = ".".join(torch.__version__.split(".")[:2])
wheel_url = f"https://download.openmmlab.com/mmcv/dist/cu{cuda_ver}/torch{torch_ver}/index.html"
print(f"Installing mmcv-full for torch {torch_ver} + CUDA {cuda_ver}")
print(f"Wheel index: {wheel_url}")
!pip install -q mmcv-full -f {wheel_url}
!pip install -q mmdet==2.20.0 torchpack nuscenes-devkit pyquaternion imageio-ffmpeg

# Compile only BEVFusion's bev_pool extension (~30s, not the full setup.py)
!cd bevfusion && python setup.py build_ext --inplace 2>&1 | tail -3

# 3 pretrained segmentation checkpoints
!mkdir -p bevfusion/pretrained
!wget -qO bevfusion/pretrained/camera-only-seg.pth  "https://www.dropbox.com/scl/fi/cwpcu80n0shmwraegi6z4/camera-only-seg.pth?rlkey=l60kdaz19fq3gwocsjk09e60z&dl=1"
!wget -qO bevfusion/pretrained/lidar-only-seg.pth   "https://www.dropbox.com/scl/fi/mi3w6uxvytdre9i42r9k7/lidar-only-seg.pth?rlkey=rve7hx80u3en1gfoi7tjucl72&dl=1"
!wget -qO bevfusion/pretrained/bevfusion-seg.pth    "https://www.dropbox.com/scl/fi/8lgd1hkod2a15mwry0fvd/bevfusion-seg.pth?rlkey=2tmgw7mcrlwy9qoqeui63tay9&dl=1"

# nuScenes mini (skip if already present from Lab 1)
![ -d nuscenes-mini/v1.0-mini ] || (mkdir -p nuscenes-mini && wget -q https://www.nuscenes.org/data/v1.0-mini.tgz -O- | tar -xz -C nuscenes-mini)
![ -d nuscenes-mini/maps/expansion ] || (wget -qO /tmp/mapexp.zip https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/nuScenes-map-expansion-v1.3.zip && unzip -qo /tmp/mapexp.zip -d nuscenes-mini/maps)

import mmcv; print(f"mmcv {mmcv.__version__} installed OK")


In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, 'bevfusion')
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from nuscenes.nuscenes import NuScenes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nusc   = NuScenes('v1.0-mini', dataroot='nuscenes-mini', verbose=False)
print(f'device: {device}')
print(f'nuScenes mini: {len(nusc.sample)} samples across {len(nusc.scene)} scenes')

## Part 2 — Load one sample: 6 cameras + LiDAR

Unlike Lab 1 (cameras only), BEVFusion also needs the **LiDAR point cloud**. A nuScenes keyframe has:
- 6 surround camera images (1600×900)
- 1 LiDAR sweep from LIDAR_TOP (~35k points, x/y/z/intensity/ring)

In [ ]:
from nuscenes.utils.data_classes import LidarPointCloud
from pyquaternion import Quaternion
import os

# Pick a sample
sample = nusc.sample[10]

# Load the 6 camera images
cam_names = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']
cam_images = {}
for cam in cam_names:
    sd = nusc.get('sample_data', sample['data'][cam])
    cam_images[cam] = Image.open(os.path.join(nusc.dataroot, sd['filename']))

# Load the LiDAR point cloud and transform to ego frame
lidar_sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
pc = LidarPointCloud.from_file(os.path.join(nusc.dataroot, lidar_sd['filename']))

calib = nusc.get('calibrated_sensor', lidar_sd['calibrated_sensor_token'])
pc.rotate(Quaternion(calib['rotation']).rotation_matrix)
pc.translate(np.array(calib['translation']))

lidar_points = pc.points.T  # (N, 5) — x, y, z, intensity, ring

print(f'Cameras: {len(cam_images)} images, each {cam_images["CAM_FRONT"].size}')
print(f'LiDAR:   {lidar_points.shape[0]:,} points')
print(f'  x range: {lidar_points[:,0].min():.1f} .. {lidar_points[:,0].max():.1f} m')
print(f'  z range: {lidar_points[:,2].min():.1f} .. {lidar_points[:,2].max():.1f} m')

In [ ]:
# Show what we have: 6 cameras + LiDAR BEV (top-down scatter of points)
fig, axes = plt.subplots(3, 3, figsize=(14, 10))

# Top row + middle row: 6 cameras
for i, cam in enumerate(cam_names):
    ax = axes[i // 3, i % 3]
    ax.imshow(cam_images[cam]); ax.axis('off')
    ax.set_title(cam, fontsize=9)

# Bottom-center: LiDAR bird's eye view
for ax in axes[2]: ax.axis('off')
ax_lid = axes[2, 1]
ax_lid.scatter(lidar_points[:, 1], lidar_points[:, 0],
               c=lidar_points[:, 2], s=0.3, cmap='turbo', vmin=-2, vmax=3)
ax_lid.set_xlim(50, -50); ax_lid.set_ylim(-50, 50)
ax_lid.set_aspect('equal')
ax_lid.set_title(f'LiDAR top-down ({lidar_points.shape[0]:,} points, colored by height)', fontsize=9)
ax_lid.scatter([0], [0], c='red', marker='^', s=100, zorder=5)

plt.tight_layout(); plt.show()

## Part 3 — Run BEVFusion on one keyframe

Two steps: first generate the `.pkl` info files that BEVFusion's dataloader expects (pre-computed metadata for each sample), then load the model and run inference.

We'll start with the **fused** checkpoint (camera + LiDAR), then compare with camera-only and LiDAR-only.

In [ ]:
# Generate the .pkl info files BEVFusion needs (metadata for each nuScenes sample).
# This runs once and takes ~30s on mini.
!cd bevfusion && python tools/create_data.py nuscenes \
  --root-path ../nuscenes-mini \
  --version v1.0-mini \
  --out-dir ../nuscenes-mini \
  --extra-tag nuscenes 2>&1 | tail -5

import os
for f in os.listdir('nuscenes-mini'):
    if f.endswith('.pkl'):
        print(f'  {f}  ({os.path.getsize(os.path.join("nuscenes-mini", f)) / 1e6:.1f} MB)')

In [ ]:
# Load BEVFusion model from config + pretrained weights
import sys; sys.path.insert(0, 'bevfusion')

from mmcv import Config
from mmdet3d.models import build_model
from mmdet3d.utils import recursive_eval
from mmcv.runner import load_checkpoint
from torchpack.utils.config import configs as C

# Load the segmentation config (camera + lidar fusion variant)
config_path = 'bevfusion/configs/nuscenes/seg/fusion-bev256d2-lss.yaml'
C.load(config_path, recursive=True)
cfg = Config(recursive_eval(C), filename=config_path)

# Build model and load fused checkpoint
model = build_model(cfg.model, test_cfg=cfg.get('test_cfg'))
load_checkpoint(model, 'bevfusion/pretrained/bevfusion-seg.pth', map_location='cpu')
model.cuda().eval()

print('BEVFusion loaded (fused segmentation)')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M')

In [ ]:
# Build the dataset and grab one preprocessed sample
from mmdet3d.datasets import build_dataset

# Point the config to our nuScenes mini
cfg.data.test.dataset_root = 'nuscenes-mini/'
cfg.data.test.ann_file = 'nuscenes-mini/nuscenes_infos_val.pkl'

dataset = build_dataset(cfg.data.test)
print(f'Dataset: {len(dataset)} samples')

# Get one preprocessed sample (returns a dict with images, points, etc.)
sample = dataset[0]
print()
print('Keys in one sample:')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k:30s} {str(v.shape):20s} {v.dtype}')
    elif hasattr(v, '__len__'):
        print(f'  {k:30s} len={len(v)}')
    else:
        print(f'  {k:30s} {type(v).__name__}')

In [ ]:
# Run inference on this one sample
from mmcv.parallel import collate, scatter

# Collate into a batch of 1 and move to GPU
batch = collate([sample], samples_per_gpu=1)
batch = scatter(batch, [device])[0]

with torch.no_grad():
    output = model(**batch)

print('Output keys:', list(output.keys()) if isinstance(output, dict) else type(output))
print()

# The segmentation output is a BEV mask with 6 map classes:
# drivable_area, ped_crossing, walkway, stop_line, carpark_area, divider
map_classes = ['drivable_area', 'ped_crossing', 'walkway', 'stop_line', 'carpark_area', 'divider']

# Extract the prediction
if isinstance(output, dict) and 'masks_bev' in output:
    bev_seg = output['masks_bev'].cpu().numpy()  # (1, num_classes, H, W) or similar
    print(f'BEV segmentation shape: {bev_seg.shape}')
else:
    print('Output structure:', output)
    print('(Adjust the key name based on what you see above)')

In [ ]:
# Visualize the BEV segmentation
colors = {
    'drivable_area': [0.9, 0.6, 0.3],
    'ped_crossing':  [1.0, 0.3, 0.3],
    'walkway':       [0.3, 0.8, 0.3],
    'stop_line':     [1.0, 1.0, 0.0],
    'carpark_area':  [0.5, 0.5, 0.9],
    'divider':       [0.9, 0.3, 0.9],
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for i, (cls_name, color) in enumerate(colors.items()):
    ax = axes[i // 3, i % 3]
    if bev_seg.ndim == 4:
        mask = bev_seg[0, i]
    else:
        mask = bev_seg[i]
    ax.imshow(mask, cmap='gray', vmin=0, vmax=1)
    ax.set_title(cls_name)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('BEVFusion segmentation output (camera + LiDAR fused)', fontsize=13)
plt.tight_layout(); plt.show()